# 🚀 Chapter 06-02b: Build a Database Agent with Tool Calling

---

## 📖 The Mission

You're building an AI assistant for a company that can query a SQLite database, analyze results, and generate reports — all through natural language.

Boss says: *"I want to ask questions in English and get answers from our data."*

Let's build it step by step!

---

## 🧪 Step 1: Setup & Create a Sample Database

In [ ]:
# !pip install langchain-openai langchain-core python-dotenv

In [ ]:
import sqlite3
import json
import os
from dotenv import load_dotenv

load_dotenv()

# Create an in-memory database with sample data
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create tables
cursor.executescript("""
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary REAL,
    hire_date TEXT
);

CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    employee_id INTEGER,
    product TEXT,
    amount REAL,
    sale_date TEXT,
    FOREIGN KEY(employee_id) REFERENCES employees(id)
);

INSERT INTO employees VALUES (1, 'Alice', 'Engineering', 120000, '2021-03-15');
INSERT INTO employees VALUES (2, 'Bob', 'Sales', 85000, '2020-07-01');
INSERT INTO employees VALUES (3, 'Charlie', 'Engineering', 110000, '2022-01-10');
INSERT INTO employees VALUES (4, 'Diana', 'Marketing', 95000, '2021-09-20');
INSERT INTO employees VALUES (5, 'Eve', 'Sales', 90000, '2023-02-14');

INSERT INTO sales VALUES (1, 2, 'Widget A', 5000, '2024-01-15');
INSERT INTO sales VALUES (2, 2, 'Widget B', 3200, '2024-01-20');
INSERT INTO sales VALUES (3, 5, 'Widget A', 7500, '2024-02-01');
INSERT INTO sales VALUES (4, 5, 'Widget C', 4100, '2024-02-10');
INSERT INTO sales VALUES (5, 2, 'Widget A', 6000, '2024-03-05');
INSERT INTO sales VALUES (6, 5, 'Widget B', 8200, '2024-03-15');
""")
conn.commit()
print("✅ Database created with employees and sales tables!")

## 🧪 Step 2: Define Tools with LangChain's @tool Decorator

In [ ]:
from langchain_core.tools import tool
from typing import Optional

@tool
def list_tables() -> str:
    """List all tables in the database with their column info."""
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    result = []
    for (table_name,) in tables:
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        col_info = [f"{c[1]} ({c[2]})" for c in columns]
        result.append(f"Table: {table_name}\n  Columns: {', '.join(col_info)}")
    return "\n".join(result)


@tool
def run_query(sql: str) -> str:
    """Execute a READ-ONLY SQL query on the database. Only SELECT statements are allowed."""
    # Safety: only allow SELECT
    sql_stripped = sql.strip().upper()
    if not sql_stripped.startswith("SELECT"):
        return "ERROR: Only SELECT queries are allowed for safety."
    try:
        cursor.execute(sql)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        if not rows:
            return "Query returned no results."
        # Format as simple table
        header = " | ".join(columns)
        lines = [header, "-" * len(header)]
        for row in rows[:50]:  # Limit to 50 rows
            lines.append(" | ".join(str(v) for v in row))
        if len(rows) > 50:
            lines.append(f"... ({len(rows)} total rows, showing first 50)")
        return "\n".join(lines)
    except Exception as e:
        return f"SQL Error: {str(e)}"


@tool
def summarize_data(data: str, summary_type: str = "overview") -> str:
    """Summarize query results. summary_type can be 'overview', 'count', or 'stats'."""
    lines = data.strip().split("\n")
    if len(lines) < 2:
        return "No data to summarize."
    data_rows = [l for l in lines[2:] if not l.startswith("...")]
    return f"Summary ({summary_type}): {len(data_rows)} data rows found."


tools = [list_tables, run_query, summarize_data]
print("✅ Tools defined:")
for t in tools:
    print(f"  🔧 {t.name}: {t.description[:60]}...")

## 🧪 Step 3: Bind Tools to LLM

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

print("✅ LLM bound with 3 tools!")

## 🧪 Step 4: Build the Agent Loop

This is the core ReAct pattern — Think → Act → Observe → Repeat

In [ ]:
# Map tool names to functions
tool_map = {t.name: t for t in tools}

SYSTEM_PROMPT = """You are a helpful database assistant. You can query a SQLite database
to answer questions about employees and sales.

Always start by listing tables to understand the schema.
Then write SQL queries to answer the user's question.
Provide clear, formatted answers with the data."""

def run_db_agent(user_question: str, max_steps: int = 8):
    """Run the DB agent with tool calling loop."""
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_question)
    ]
    
    print(f"\n{'='*60}")
    print(f"🧑 User: {user_question}")
    print(f"{'='*60}")
    
    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # No tool calls → final answer
        if not response.tool_calls:
            print(f"\n🤖 Final Answer:\n{response.content}")
            return response.content
        
        # Execute each tool call
        for tc in response.tool_calls:
            print(f"\n🔧 Step {step+1}: Calling {tc['name']}({tc['args']})")
            
            if tc["name"] not in tool_map:
                result = f"Error: Unknown tool '{tc['name']}'"
            else:
                try:
                    result = tool_map[tc["name"]].invoke(tc["args"])
                except Exception as e:
                    result = f"Error: {str(e)}"
            
            print(f"   📊 Result: {str(result)[:200]}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    
    print("⚠️ Max steps reached!")
    return "Max steps reached."

print("✅ Agent loop ready!")

## 🧪 Step 5: Test the Agent!

In [ ]:
# Test 1: Simple query
run_db_agent("How many employees are in each department?")

In [ ]:
# Test 2: Multi-step reasoning
run_db_agent("Who is the top salesperson by total sales amount? Show their details.")

In [ ]:
# Test 3: Complex analysis
run_db_agent("Compare average salary between departments. Which department has the highest?")

## 🚀 Step 6: Enhanced Agent with Conversation Memory

In [ ]:
class DBChatAgent:
    """Database agent with conversation memory."""
    
    def __init__(self, tools, max_steps=8):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)
        self.tool_map = {t.name: t for t in tools}
        self.max_steps = max_steps
        self.history = [
            SystemMessage(content=SYSTEM_PROMPT)
        ]
    
    def chat(self, user_input: str) -> str:
        self.history.append(HumanMessage(content=user_input))
        print(f"\n🧑 You: {user_input}")
        
        for step in range(self.max_steps):
            response = self.llm.invoke(self.history)
            self.history.append(response)
            
            if not response.tool_calls:
                print(f"🤖 Agent: {response.content}")
                return response.content
            
            for tc in response.tool_calls:
                print(f"  🔧 [{tc['name']}] {tc['args']}")
                if tc["name"] in self.tool_map:
                    try:
                        result = self.tool_map[tc["name"]].invoke(tc["args"])
                    except Exception as e:
                        result = f"Error: {str(e)}"
                else:
                    result = f"Unknown tool: {tc['name']}"
                self.history.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
        
        return "Max steps reached."

# Create agent & have a multi-turn conversation
agent = DBChatAgent(tools)

agent.chat("What tables do we have?")

In [ ]:
# Follow-up (agent remembers context!)
agent.chat("Show me all employees in Engineering")

In [ ]:
# Another follow-up referencing earlier context
agent.chat("What are their total sales?")

## ✍️ Practice Challenge

**Add a new tool** called `generate_report` that takes a `title` and `data` and formats them into a clean Markdown report.

Then ask the agent: *"Generate a monthly sales report for Q1 2024"*

<details>
<summary>💡 Hint</summary>

```python
@tool
def generate_report(title: str, data: str) -> str:
    """Generate a formatted Markdown report with title and data."""
    return f"# {title}\n\n{data}\n\n---\n*Report generated by DB Agent*"
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 @tool decorator makes any Python function LLM-callable.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 .bind_tools() attaches tool schemas to the LLM.                │
├─────────────────────────────────────────────────────────────────────┤
│  📌 The agent loop: Invoke → Check tool_calls → Execute → Repeat.  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Always validate SQL — restrict to SELECT, limit rows.          │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Conversation memory = keep appending messages to history.       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Chapter 07 — LangGraph Fundamentals!

---

### 🎉 You Built a Database Agent!
From natural language to SQL results — that's real power. 🚀